In [1]:
# Installing Dependencies
!pip install -qU langchain langchain-community langchain-text-splitters langchain-google-genai pypdf faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires request

In [2]:
# Imports
import os
from google.colab import userdata, files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# Securely fetch the API key from Colab Secrets
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')
    print("✅ API Key loaded successfully.")
except Exception as e:
    print("❌ Error: Please set GOOGLE_API_KEY in the Colab Secrets tab.")

✅ API Key loaded successfully.


In [5]:
# Cell 3
import os

# Define the exact name of the file you uploaded to Colab
pdf_filename = "intro-to-ml.pdf"

# Colab's default working directory is /content/
pdf_path = f"/content/{pdf_filename}"

# Verify the file exists before proceeding
if os.path.exists(pdf_path):
    print(f"✅ File successfully located at: {pdf_path}")
    print(f"File size: {os.path.getsize(pdf_path) / (1024*1024):.2f} MB")
else:
    print(f"❌ Error: Could not find '{pdf_filename}'.")
    print("Please ensure you dragged and dropped the file into the Files panel on the left.")

✅ File successfully located at: /content/intro-to-ml.pdf
File size: 6.73 MB


In [3]:
!pip install -qU langchain-huggingface sentence-transformers

In [4]:
!pip install -qU pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 58.7 MB/s eta 0:00:00


In [10]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings


loader = PyMuPDFLoader(pdf_path)
pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(pages)
print(f" Created {len(chunks)} chunks.")



 Created 663 chunks.


In [11]:
# Using HuggingFace  Embedding Model
print("\n2. Initializing HuggingFace Embedding Model...")
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("\n3. Running Quarantine Embedding Loop...")
valid_texts = []
valid_metadatas = []
valid_embeddings = []

# Process chunks one by one. If one crashes, we catch the error and skip it!
for i, doc in enumerate(chunks):
    text = str(doc.page_content).replace('\x00', '').replace('\ufffd', '').strip()

    if len(text) < 15:
        continue

    try:
        # Attempt to embed just this single piece of text
        emb = embeddings_model.embed_query(text)

        # If it succeeds, save everything
        valid_texts.append(text)
        valid_embeddings.append(emb)
        valid_metadatas.append(doc.metadata)

    except Exception as e:
        # IF IT CRASHES, we catch it here instead of breaking the notebook
        print(f"⚠️ Caught and skipped poisoned chunk at index {i}!")

print(f"\n✅ Successfully embedded {len(valid_texts)} safe chunks!")

print("\n4. Building and saving FAISS index...")
# We use from_embeddings because we already generated the math vectors safely
text_embeddings = list(zip(valid_texts, valid_embeddings))
vectorstore = FAISS.from_embeddings(text_embeddings, embeddings_model, metadatas=valid_metadatas)

vectorstore.save_local("faiss_huggingface_index")
print("💾 FAISS index saved securely as 'faiss_huggingface_index'!")


2. Initializing HuggingFace Embedding Model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



3. Running Quarantine Embedding Loop...

✅ Successfully embedded 663 safe chunks!

4. Building and saving FAISS index...
💾 FAISS index saved securely as 'faiss_huggingface_index'!


In [8]:
# 5. Setup Retriever and Gemini LLM

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})


from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0)

system_prompt = (
    "You are an expert Machine Learning assistant."
    "Use the following pieces of retrieved context to answer the question."
    "If you don't know the answer or if the answer is not contained in the context, "
    "just say that you don't know. Do NOT make up an answer or use outside knowledge."
    "Keep the answer concise and strictly factual.\n\n"
    "Context: {context}"
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [9]:
# END-TO-END RAG Q&A APPLICATION

while True:
    # 1. Accept user query
    query = input("\nAsk a question about Machine Learning: ")

    if query.lower() in ['exit', 'quit']:
        print("Goodbye!")
        break

    if not query.strip():
        continue

    print("\n🔍 Searching textbook and thinking...")

    # Step 1: Retrieve the top-k most relevant chunks from FAISS
    retrieved_docs = retriever.invoke(query)

    # Step 2: Stitch the retrieved text together into one big "Context" string
    context_text = "\n\n---\n\n".join([doc.page_content for doc in retrieved_docs])

    # Step 3: Inject the context and the user's query into our strict prompt template
    formatted_prompt = prompt_template.format_messages(
        context=context_text,
        input=query
    )

    # Step 4: Send the prompt to the LLM
    llm_response = llm.invoke(formatted_prompt)

    # Display the generated answer
    print("\n💡 Answer:")
    print(llm_response.content)

    #Evidence of grounding (Show the sources)
    print("📚 Evidence / Retrieved Context")
    print("-"*40)
    for i, doc in enumerate(retrieved_docs):
        # Extract page number
        page_num = doc.metadata.get("page", "Unknown")

        snippet = doc.page_content[:200].replace('\n', ' ')

        print(f"🔹 Source {i+1} (Page {page_num}): {snippet}...\n")


Ask a question about Machine Learning: What is the author name

🔍 Searching textbook and thinking...

💡 Answer:
The authors are Andreas and Sarah.
📚 Evidence / Retrieved Context
----------------------------------------
🔹 Source 1 (Page 12): I want to thank my reviewers, Thomas Caswell, Olivier Grisel, Stefan van der Walt, and John Myles White, who took the time to read the early versions of this book and provided me with invaluable feedb...

🔹 Source 2 (Page 11): Safari® Books Online Safari Books Online is an on-demand digital library that deliv‐ ers expert content in both book and video form from the world’s leading authors in technology and business. Technol...

🔹 Source 3 (Page 13): undertake such a challenging task. From Sarah I would like to thank Meg Blanchette, without whose help and guidance this project would not have even existed. Thanks to Celia La and Brian Carlson for r...

🔹 Source 4 (Page 391): activity near their habitat means greater amounts of sediment and chemicals 

### Project Summary

This project implements a Retrieval Augmented Generation (RAG) Q&A application within Google Colab. It processes a PDF document, "intro-to-ml.pdf", to create a searchable knowledge base capable of answering questions related to Machine Learning.

Here are the specifics:

*   **Embedding Model Used**: The project utilizes the `HuggingFaceEmbeddings` model, specifically `all-MiniLM-L6-v2`, to convert text chunks from the PDF into numerical vector representations.

*   **Vector Store Used**: The `FAISS` (Facebook AI Similarity Search) library is employed as the vector store to efficiently store and retrieve these embeddings. The index was saved locally as `faiss_huggingface_index`.

*   **Language Model (LLM) Used**: The `ChatGoogleGenerativeAI` model, specifically `gemini-2.5-flash`, is used for generating responses based on the retrieved context.

*   **Last Answered Query**: The last meaningful query processed by the RAG application was: "what are websites i can go to get the datasets for learning machine learning".